# VideoDB Understanding Quickstart

Create an analyzer-based Understanding run, wait for completion, and inspect analyzer outputs.

Understanding answers: **what is in the video?** Indexing and search are covered in the Indexing V2 notebooks.


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 1. Install dependencies

In [ ]:
!pip install -q --force-reinstall --no-cache-dir "git+https://github.com/video-db/videodb-python.git@feat/add-indexing-v2" python-dotenv


## 2. Connect to VideoDB

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
print("Connected to VideoDB")


## 3. Choose a video

By default, this notebook uploads the sample video used in the E2E flow: **Silicon Valley - Gilfoyle is free for hire**. To use an existing video instead, comment the upload line and uncomment the `get_video` lines in the next cell.


In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

collection = conn.get_collection()
video = collection.upload(VIDEO_URL)

# To use an existing video instead, comment the upload line above and uncomment these lines:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Collection:", collection.id)
print("Video:", video.id)
video.play()


## 4. Create an Understanding run

This run creates three analyzers:

- `transcript` from spoken words
- `objects` from object detection
- `scene` from VLM with transcript and object context


In [ ]:
OBJECT_LABELS = [
    "person",
    "cup",
    "bottle",
    "chair",
    "sofa",
    "diningtable",
    "laptop",
    "cell phone",
    "book",
]


In [ ]:
understanding = video.understand(
    analyzers=[
        {
            "type": "spoken_words",
            "name": "transcript",
            "config": {"language": "en"},
        },
        {
            "type": "object_detection",
            "name": "objects",
            "sampling": {"strategy": "interval", "every": 1},
            "config": {
                "labels": OBJECT_LABELS,
                "confidence_threshold": 0.35,
                "include_bounding_boxes": True,
            },
        },
        {
            "type": "vlm",
            "name": "scene",
            "inputs": ["transcript", "objects"],
            "sampling": {"strategy": "uniform", "frame_count": 3},
            "config": {
                "model": "ultra",
                "prompt": (
                    "Analyze this Silicon Valley clip temporally from the sampled frame sequence. "
                    "Use the detected objects and transcript when helpful. "
                    "Describe what happens in the scene and return it in the outputs field."
                ),
                "schema": {"outputs": "text"},
            },
        },
    ],
    segmentation={"type": "shot", "threshold": 30},
)

print("Understanding ID:", understanding.id)
print("Status:", understanding.status)
understanding.list_analyzers()


## 5. Check status

In [ ]:
understanding.refresh()
print("Status:", understanding.status)

for analyzer in understanding.list_analyzers():
    print(analyzer.name, analyzer.type, analyzer.status)


## 6. Wait until complete

In [ ]:
understanding.wait_until_complete(timeout=3600, poll_interval=15)
print("Final status:", understanding.status)

for analyzer in understanding.list_analyzers():
    print(analyzer.name, analyzer.type, analyzer.status)


## 7. Fetch analyzer outputs

In [ ]:
def as_segments(output):
    """Handle both list outputs and {'scenes': [...]} outputs."""
    return output.get("scenes", output) if isinstance(output, dict) else output


def preview_segments(name, output, max_segments=3):
    segments = as_segments(output) or []
    print(f"{name}: {len(segments)} segments")
    print("=" * 60)
    for segment in segments[:max_segments]:
        print(f"{segment.get('start')}s → {segment.get('end')}s")
        print(segment.get("data"))
        print("-" * 60)


In [ ]:
transcript = understanding.get_analyzer("transcript").get_output()
objects = understanding.get_analyzer("objects").get_output()
scene = understanding.get_analyzer("scene").get_output()

preview_segments("transcript", transcript)
preview_segments("objects", objects)
preview_segments("scene", scene)


## 8. Work with existing Understanding runs

In [ ]:
same_understanding = video.get_understanding(understanding.id)
print(same_understanding)
same_understanding.list_analyzers()


In [ ]:
for item in video.list_understandings():
    print(item.id, item.status)


## 9. Optional: wait for one analyzer

In [ ]:
transcript_analyzer = understanding.get_analyzer("transcript")
transcript_analyzer.wait_until_complete(timeout=900, poll_interval=10)
print(transcript_analyzer.name, transcript_analyzer.status)


## 10. Optional cleanup

In [ ]:
DELETE_UNDERSTANDING = False

if DELETE_UNDERSTANDING:
    understanding.delete()
    print("Deleted", understanding.id)
else:
    print("Skipping delete. Set DELETE_UNDERSTANDING=True to delete this Understanding.")


## Method map

```text
video.understand(...)
understanding.refresh()
understanding.wait_until_complete(...)
understanding.list_analyzers()
understanding.get_analyzer(...).get_output()
video.get_understanding(...)
video.list_understandings()
understanding.delete()
```
